In [1]:
### Imports ###

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import json
from sklearn.covariance import LedoitWolf

def load_feature_order(path: Path):
    """Load feature order from a JSON file."""
    import json
    with open(path, 'r') as f:
        data = json.load(f)
    # Handle both list and dict formats
    if isinstance(data, list):
        return data
    elif isinstance(data, dict) and 'features' in data:
        return data['features']
    elif isinstance(data, dict) and 'feature_order' in data:
        return data['feature_order']
    else:
        raise ValueError(f"Unexpected format in {path}: expected list or dict with 'features'/'feature_order' key")

In [2]:
### Load datasets ###
# Note: coral stats estimated from train splits only

# Load source training data
source_train_path = Path("data/processed/source/source_train.csv")
source_train_df = pd.read_csv(source_train_path)

# Load target training data
target_train_path = Path("data/processed/target/target_train.csv")
target_train_df = pd.read_csv(target_train_path)

# Separate features and labels (assuming last column is the label)
X_source_train = source_train_df.iloc[:, :-1]
y_source_train = source_train_df.iloc[:, -1]

X_target_train = target_train_df.iloc[:, :-1]
y_target_train = target_train_df.iloc[:, -1]

print(f"Source training data: {X_source_train.shape[0]} samples, {X_source_train.shape[1]} features")
print(f"Target training data: {X_target_train.shape[0]} samples, {X_target_train.shape[1]} features")
print(f"\nSource label distribution:\n{y_source_train.value_counts().sort_index()}")
print(f"\nTarget label distribution:\n{y_target_train.value_counts().sort_index()}")

Source training data: 1259420 samples, 77 features
Target training data: 2733516 samples, 77 features

Source label distribution:
Label
0    944565
1    102420
2      8234
3    184099
4      4399
5      4637
6      6348
7      4718
Name: count, dtype: int64

Target label distribution:
Label
0    2050137
1     161384
2      33206
3     240000
4      80000
5       8792
6      79997
7      80000
Name: count, dtype: int64


In [3]:
### Verify feature order and label space ###

# Load shared feature space contract
shared_feature_space_path = Path("data/processed/shared_feature_space.json")
if not shared_feature_space_path.exists():
    raise FileNotFoundError(f"Shared feature space file not found at {shared_feature_space_path}")

shared_features = load_feature_order(shared_feature_space_path)

# Load shared label space contract
shared_label_space_path = Path("data/processed/shared_label_space.json")
if not shared_label_space_path.exists():
    raise FileNotFoundError(f"Shared label space file not found at {shared_label_space_path}")

with open(shared_label_space_path, "r", encoding="utf-8") as f:
    label_payload = json.load(f)

if isinstance(label_payload, dict):
    if "labels" not in label_payload:
        raise ValueError(f"Expected key 'labels' in {shared_label_space_path}")
    shared_labels = list(label_payload["labels"])
elif isinstance(label_payload, list):
    shared_labels = list(label_payload)
else:
    raise ValueError(f"Unexpected format in {shared_label_space_path}: {type(label_payload).__name__}")

if not shared_labels:
    raise ValueError(f"Shared label space in {shared_label_space_path} is empty")

# Load label encoder so labels can be interpreted as both encoded IDs and class names.
label_encoder_path = Path("models/label_encoder.joblib")
if not label_encoder_path.exists():
    raise FileNotFoundError(f"Label encoder file not found at {label_encoder_path}")

le = joblib.load(label_encoder_path)
if not hasattr(le, "classes_") or len(le.classes_) == 0:
    raise ValueError("Loaded label encoder does not contain classes_")

encoder_classes = [str(c) for c in le.classes_]
encoded_to_name = {idx: name for idx, name in enumerate(encoder_classes)}
name_to_encoded = {name: idx for idx, name in encoded_to_name.items()}

def resolve_label_set(values, context_name):
    """Resolve labels to both encoded IDs and class names using the label encoder."""
    resolved_encoded = set()
    resolved_names = set()
    unresolved = []

    for raw in values:
        encoded_candidate = None

        if isinstance(raw, (int, np.integer)):
            encoded_candidate = int(raw)
        elif isinstance(raw, (float, np.floating)) and float(raw).is_integer():
            encoded_candidate = int(raw)
        elif isinstance(raw, str):
            raw_str = raw.strip()
            if raw_str.lstrip("-").isdigit():
                encoded_candidate = int(raw_str)

        if encoded_candidate is not None and encoded_candidate in encoded_to_name:
            resolved_encoded.add(encoded_candidate)
            resolved_names.add(encoded_to_name[encoded_candidate])
            continue

        name_candidate = str(raw)
        if name_candidate in name_to_encoded:
            resolved_names.add(name_candidate)
            resolved_encoded.add(name_to_encoded[name_candidate])
            continue

        unresolved.append(raw)

    if unresolved:
        raise ValueError(
            f"{context_name} contains labels not recognized by label encoder: {sorted(map(str, unresolved))}"
        )

    return resolved_encoded, resolved_names

# Verify source dataset feature space
source_features_present = set(X_source_train.columns)
source_features_expected = set(shared_features)
source_features_missing = source_features_expected - source_features_present
source_features_extra = source_features_present - source_features_expected

if source_features_missing:
    raise ValueError(
        f"Source dataset is missing features: {sorted(source_features_missing)}"
    )
if source_features_extra:
    raise ValueError(
        f"Source dataset has unexpected features: {sorted(source_features_extra)}"
    )

# Verify target dataset feature space
target_features_present = set(X_target_train.columns)
target_features_expected = set(shared_features)
target_features_missing = target_features_expected - target_features_present
target_features_extra = target_features_present - target_features_expected

if target_features_missing:
    raise ValueError(
        f"Target dataset is missing features: {sorted(target_features_missing)}"
    )
if target_features_extra:
    raise ValueError(
        f"Target dataset has unexpected features: {sorted(target_features_extra)}"
    )

# Resolve label spaces using label encoder
shared_labels_encoded, shared_labels_names = resolve_label_set(shared_labels, "Shared label space")
source_labels_present_raw = set(y_source_train.unique())
source_labels_encoded, source_labels_names = resolve_label_set(
    source_labels_present_raw,
    "Source dataset labels",
)
target_labels_present_raw = set(y_target_train.unique())
target_labels_encoded, target_labels_names = resolve_label_set(
    target_labels_present_raw,
    "Target dataset labels",
)

source_labels_invalid = source_labels_encoded - shared_labels_encoded
target_labels_invalid = target_labels_encoded - shared_labels_encoded

if source_labels_invalid:
    invalid_name_pairs = [
        (label_id, encoded_to_name[label_id]) for label_id in sorted(source_labels_invalid)
    ]
    raise ValueError(
        "Source dataset contains labels not in shared label space: "
        f"{invalid_name_pairs}"
    )

if target_labels_invalid:
    invalid_name_pairs = [
        (label_id, encoded_to_name[label_id]) for label_id in sorted(target_labels_invalid)
    ]
    raise ValueError(
        "Target dataset contains labels not in shared label space: "
        f"{invalid_name_pairs}"
    )

# Print verification summary
print("✓ Feature space validation passed")
print(f"  Shared feature count: {len(shared_features)}")
print(f"  Source dataset features: {X_source_train.shape[1]} (✓ matches)")
print(f"  Target dataset features: {X_target_train.shape[1]} (✓ matches)")

print("\n✓ Label encoder loaded")
print(f"  Label encoder path: {label_encoder_path}")
print(f"  Label encoder classes: {encoder_classes}")

print("\n✓ Label space validation passed")
print(f"  Shared labels (encoded -> name): {[(i, encoded_to_name[i]) for i in sorted(shared_labels_encoded)]}")
print(f"  Source labels present (encoded -> name): {[(i, encoded_to_name[i]) for i in sorted(source_labels_encoded)]}")
print(f"  Target labels present (encoded -> name): {[(i, encoded_to_name[i]) for i in sorted(target_labels_encoded)]}")

print("\n✓ All verification checks passed - datasets are compatible with shared contracts")

✓ Feature space validation passed
  Shared feature count: 77
  Source dataset features: 77 (✓ matches)
  Target dataset features: 77 (✓ matches)

✓ Label encoder loaded
  Label encoder path: models/label_encoder.joblib
  Label encoder classes: ['Benign', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'SSH-Patator']

✓ Label space validation passed
  Shared labels (encoded -> name): [(0, 'Benign'), (1, 'DDoS'), (2, 'DoS GoldenEye'), (3, 'DoS Hulk'), (4, 'DoS Slowhttptest'), (5, 'DoS slowloris'), (6, 'FTP-Patator'), (7, 'SSH-Patator')]
  Source labels present (encoded -> name): [(0, 'Benign'), (1, 'DDoS'), (2, 'DoS GoldenEye'), (3, 'DoS Hulk'), (4, 'DoS Slowhttptest'), (5, 'DoS slowloris'), (6, 'FTP-Patator'), (7, 'SSH-Patator')]
  Target labels present (encoded -> name): [(0, 'Benign'), (1, 'DDoS'), (2, 'DoS GoldenEye'), (3, 'DoS Hulk'), (4, 'DoS Slowhttptest'), (5, 'DoS slowloris'), (6, 'FTP-Patator'), (7, 'SSH-Patator')]

✓ All verification ch

In [4]:
### Pre-estimation data sanitization ###

import warnings
from sklearn.covariance import LedoitWolf as _SklearnLedoitWolf

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

# Winsorization is applied only to the copies used for CORAL stats estimation.
# This reduces the effect of extreme scaled values on covariance estimation.
WINSOR_LOWER_Q = 0.01
WINSOR_UPPER_Q = 0.99

# Features with near-zero variance in either source or target after winsorization
# are excluded from CORAL covariance estimation.
NEAR_ZERO_VARIANCE_THRESHOLD = 1e-8

# Explicit ridge added to Ledoit-Wolf covariance matrices.
CORAL_COVARIANCE_RIDGE = 1e-2

# Diagnostic thresholds.
# These thresholds are intentionally conservative for scaled data.
EXTREME_MAX_ABS_THRESHOLD = 1_000.0
EXTREME_FEATURE_STD_THRESHOLD = 100.0
MAX_CONDITION_NUMBER = 1e6

# Set these to True if you want the notebook to stop instead of warn.
FAIL_ON_EXTREME_VALUES = False
FAIL_ON_HIGH_CONDITION_NUMBER = False

# Output diagnostics.
SANITIZATION_OUTPUT_DIR = Path("models/coral_sanitization_diagnostics")
SANITIZATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def _require_numeric_and_finite(name: str, df: pd.DataFrame) -> None:
    """Fail if any feature column is non-numeric or contains NaN/inf."""
    non_numeric = df.columns[~df.dtypes.apply(lambda dt: np.issubdtype(dt, np.number))].tolist()
    if non_numeric:
        raise TypeError(f"{name} contains non-numeric columns: {non_numeric}")

    arr = df.to_numpy(dtype=np.float64)
    nonfinite_count = int((~np.isfinite(arr)).sum())
    if nonfinite_count > 0:
        raise ValueError(f"{name} contains {nonfinite_count:,} non-finite values.")


def _feature_diagnostics(name: str, df: pd.DataFrame) -> pd.DataFrame:
    """Return feature-level diagnostics for one dataset."""
    arr = df.to_numpy(dtype=np.float64)

    diagnostics = pd.DataFrame({
        "dataset": name,
        "feature": df.columns,
        "mean": df.mean(axis=0).to_numpy(),
        "std": df.std(axis=0).to_numpy(),
        "var": df.var(axis=0).to_numpy(),
        "min": df.min(axis=0).to_numpy(),
        "q001": df.quantile(0.001).to_numpy(),
        "q01": df.quantile(0.01).to_numpy(),
        "median": df.median(axis=0).to_numpy(),
        "q99": df.quantile(0.99).to_numpy(),
        "q999": df.quantile(0.999).to_numpy(),
        "max": df.max(axis=0).to_numpy(),
        "max_abs": df.abs().max(axis=0).to_numpy(),
        "nonfinite_count": (~np.isfinite(arr)).sum(axis=0),
    })

    diagnostics["near_zero_variance"] = diagnostics["var"] < NEAR_ZERO_VARIANCE_THRESHOLD
    diagnostics["extreme_max_abs"] = diagnostics["max_abs"] > EXTREME_MAX_ABS_THRESHOLD
    diagnostics["extreme_std"] = diagnostics["std"] > EXTREME_FEATURE_STD_THRESHOLD

    return diagnostics


def _print_dataset_summary(name: str, df: pd.DataFrame, diagnostics: pd.DataFrame) -> None:
    """Print dataset-level and top-feature diagnostics."""
    print(f"\n{name} diagnostics")
    print("-" * 80)
    print(f"shape: {df.shape}")
    print(f"global min / max: {df.min().min():.6f} / {df.max().max():.6f}")
    print(f"global mean / std: {df.to_numpy(dtype=np.float64).mean():.6f} / {df.to_numpy(dtype=np.float64).std():.6f}")
    print(f"mean abs / max abs: {df.abs().mean().mean():.6f} / {df.abs().max().max():.6f}")
    print(f"feature std min / median / max: "
          f"{diagnostics['std'].min():.6f} / "
          f"{diagnostics['std'].median():.6f} / "
          f"{diagnostics['std'].max():.6f}")
    print(f"near-zero variance features: {int(diagnostics['near_zero_variance'].sum())}")
    print(f"features over max_abs threshold ({EXTREME_MAX_ABS_THRESHOLD:g}): "
          f"{int(diagnostics['extreme_max_abs'].sum())}")
    print(f"features over std threshold ({EXTREME_FEATURE_STD_THRESHOLD:g}): "
          f"{int(diagnostics['extreme_std'].sum())}")

    print("\nTop 10 features by max_abs:")
    print(
        diagnostics.sort_values("max_abs", ascending=False)
        [["feature", "max_abs", "min", "max", "std"]]
        .head(10)
        .to_string(index=False)
    )

    print("\nTop 10 features by std:")
    print(
        diagnostics.sort_values("std", ascending=False)
        [["feature", "std", "min", "max", "max_abs"]]
        .head(10)
        .to_string(index=False)
    )


def _warn_or_fail(message: str, fail: bool) -> None:
    """Warn or fail depending on configuration."""
    if fail:
        raise ValueError(message)
    warnings.warn(message)


def _winsorize_df(df: pd.DataFrame, lower_q: float, upper_q: float) -> pd.DataFrame:
    """Winsorize each feature independently using its own quantiles."""
    lower = df.quantile(lower_q)
    upper = df.quantile(upper_q)
    return df.clip(lower=lower, upper=upper, axis=1)


def _estimate_covariance_condition_number(name: str, df: pd.DataFrame) -> dict:
    """
    Estimate covariance condition number using the same ridge-regularized
    Ledoit-Wolf logic that downstream cells will use after this cell.
    """
    X = df.to_numpy(dtype=np.float64)
    mean = X.mean(axis=0)
    X_centered = X - mean

    estimator = _SklearnLedoitWolf()
    estimator.fit(X_centered)

    cov_before_ridge = np.asarray(estimator.covariance_, dtype=np.float64)
    cov_before_ridge = (cov_before_ridge + cov_before_ridge.T) / 2.0

    cov_after_ridge = cov_before_ridge + CORAL_COVARIANCE_RIDGE * np.eye(cov_before_ridge.shape[0])
    cov_after_ridge = (cov_after_ridge + cov_after_ridge.T) / 2.0

    eigvals_before = np.linalg.eigvalsh(cov_before_ridge)
    eigvals_after = np.linalg.eigvalsh(cov_after_ridge)

    cond_before = float(np.linalg.cond(cov_before_ridge))
    cond_after = float(np.linalg.cond(cov_after_ridge))

    row = {
        "dataset": name,
        "features": df.shape[1],
        "covariance_estimator": "LedoitWolf + explicit ridge",
        "ledoit_wolf_shrinkage": float(estimator.shrinkage_),
        "ridge": CORAL_COVARIANCE_RIDGE,
        "min_eig_before_ridge": float(eigvals_before.min()),
        "max_eig_before_ridge": float(eigvals_before.max()),
        "condition_number_before_ridge": cond_before,
        "min_eig_after_ridge": float(eigvals_after.min()),
        "max_eig_after_ridge": float(eigvals_after.max()),
        "condition_number_after_ridge": cond_after,
    }

    print(f"\n{name} covariance diagnostics")
    print("-" * 80)
    print(f"Ledoit-Wolf shrinkage: {row['ledoit_wolf_shrinkage']:.6f}")
    print(f"Ridge added: {CORAL_COVARIANCE_RIDGE:.6e}")
    print(f"Eigenvalues before ridge: min={row['min_eig_before_ridge']:.6e}, "
          f"max={row['max_eig_before_ridge']:.6e}")
    print(f"Condition number before ridge: {cond_before:.6e}")
    print(f"Eigenvalues after ridge: min={row['min_eig_after_ridge']:.6e}, "
          f"max={row['max_eig_after_ridge']:.6e}")
    print(f"Condition number after ridge: {cond_after:.6e}")

    if cond_after > MAX_CONDITION_NUMBER:
        _warn_or_fail(
            f"{name} covariance condition number after ridge is still high: "
            f"{cond_after:.6e} > {MAX_CONDITION_NUMBER:.6e}",
            FAIL_ON_HIGH_CONDITION_NUMBER,
        )

    return row


# ---------------------------------------------------------------------
# 1. Enforce shared feature order before diagnostics/sanitization
# ---------------------------------------------------------------------

feature_order_original = list(shared_features)

X_source_for_coral = X_source_train[feature_order_original].copy()
X_target_for_coral = X_target_train[feature_order_original].copy()

_require_numeric_and_finite("Source training features", X_source_for_coral)
_require_numeric_and_finite("Target training features", X_target_for_coral)

print("Pre-estimation feature-space check")
print("-" * 80)
print(f"Original shared feature count: {len(feature_order_original)}")
print(f"Source features: {X_source_for_coral.shape}")
print(f"Target features: {X_target_for_coral.shape}")
print(f"Column order identical: {list(X_source_for_coral.columns) == list(X_target_for_coral.columns)}")


# ---------------------------------------------------------------------
# 2. Print and save pre-sanitization feature diagnostics
# ---------------------------------------------------------------------

source_diag_before = _feature_diagnostics("source_before_sanitization", X_source_for_coral)
target_diag_before = _feature_diagnostics("target_before_sanitization", X_target_for_coral)

_print_dataset_summary("Source before sanitization", X_source_for_coral, source_diag_before)
_print_dataset_summary("Target before sanitization", X_target_for_coral, target_diag_before)

feature_diag_before = pd.concat([source_diag_before, target_diag_before], ignore_index=True)
feature_diag_before.to_csv(
    SANITIZATION_OUTPUT_DIR / "feature_diagnostics_before_sanitization.csv",
    index=False,
)

extreme_feature_rows = feature_diag_before[
    feature_diag_before["extreme_max_abs"] | feature_diag_before["extreme_std"]
].copy()

if not extreme_feature_rows.empty:
    extreme_feature_rows.to_csv(
        SANITIZATION_OUTPUT_DIR / "flagged_extreme_features_before_sanitization.csv",
        index=False,
    )

    _warn_or_fail(
        f"Extreme scaled values were detected before covariance estimation. "
        f"Flagged rows saved to "
        f"{SANITIZATION_OUTPUT_DIR / 'flagged_extreme_features_before_sanitization.csv'}",
        FAIL_ON_EXTREME_VALUES,
    )


# ---------------------------------------------------------------------
# 3. Winsorize source/target copies used for CORAL covariance estimation
# ---------------------------------------------------------------------

print("\nApplying winsorization for CORAL covariance estimation")
print("-" * 80)
print(f"Winsorization quantiles: lower={WINSOR_LOWER_Q}, upper={WINSOR_UPPER_Q}")

X_source_winsorized = _winsorize_df(
    X_source_for_coral,
    lower_q=WINSOR_LOWER_Q,
    upper_q=WINSOR_UPPER_Q,
)

X_target_winsorized = _winsorize_df(
    X_target_for_coral,
    lower_q=WINSOR_LOWER_Q,
    upper_q=WINSOR_UPPER_Q,
)


# ---------------------------------------------------------------------
# 4. Check near-zero variance features, but keep all features
# ---------------------------------------------------------------------

source_var = X_source_winsorized.var(axis=0)
target_var = X_target_winsorized.var(axis=0)

near_zero_source = source_var[source_var < NEAR_ZERO_VARIANCE_THRESHOLD]
near_zero_target = target_var[target_var < NEAR_ZERO_VARIANCE_THRESHOLD]

near_zero_features = sorted(
    set(near_zero_source.index) | set(near_zero_target.index)
)

print("\nNear-zero variance feature check")
print("-" * 80)
print(f"Near-zero variance features in source: {len(near_zero_source)}")
print(f"Near-zero variance features in target: {len(near_zero_target)}")
print(f"Near-zero variance features in either dataset: {len(near_zero_features)}")

if near_zero_features:
    near_zero_df = pd.DataFrame({
        "feature": near_zero_features,
        "source_variance_after_winsorization": source_var.loc[near_zero_features].to_numpy(),
        "target_variance_after_winsorization": target_var.loc[near_zero_features].to_numpy(),
    })

    near_zero_output_path = (
        SANITIZATION_OUTPUT_DIR
        / "near_zero_variance_features_flagged_not_dropped.csv"
    )

    near_zero_df.to_csv(near_zero_output_path, index=False)

    print("\nFlagged near-zero variance features, not dropped:")
    print(near_zero_df.to_string(index=False))

    warnings.warn(
        "Near-zero variance features were detected. They were NOT dropped, "
        "so the covariance matrices keep the expected feature-space shape. "
        "Stability will instead be handled using winsorization plus explicit "
        "ridge regularization."
    )

# Preserve all original features to avoid breaking downstream CORAL shape expectations.
features_kept = list(feature_order_original)
features_dropped = []

X_source_sanitized = X_source_winsorized[features_kept].copy()
X_target_sanitized = X_target_winsorized[features_kept].copy()

print("\nFeature retention summary")
print("-" * 80)
print(f"Original feature count: {len(feature_order_original)}")
print(f"Features retained for CORAL stats: {len(features_kept)}")
print(f"Features dropped: {len(features_dropped)}")
print(f"Covariance matrix shape will remain: {len(features_kept)} x {len(features_kept)}")


# ---------------------------------------------------------------------
# 5. Print and save post-sanitization feature diagnostics
# ---------------------------------------------------------------------

source_diag_after = _feature_diagnostics("source_after_sanitization", X_source_sanitized)
target_diag_after = _feature_diagnostics("target_after_sanitization", X_target_sanitized)

_print_dataset_summary("Source after sanitization", X_source_sanitized, source_diag_after)
_print_dataset_summary("Target after sanitization", X_target_sanitized, target_diag_after)

feature_diag_after = pd.concat([source_diag_after, target_diag_after], ignore_index=True)
feature_diag_after.to_csv(
    SANITIZATION_OUTPUT_DIR / "feature_diagnostics_after_sanitization.csv",
    index=False,
)


# ---------------------------------------------------------------------
# 6. Estimate covariance condition numbers after sanitization
# ---------------------------------------------------------------------

cov_diag_rows = [
    _estimate_covariance_condition_number("source_after_sanitization", X_source_sanitized),
    _estimate_covariance_condition_number("target_after_sanitization", X_target_sanitized),
]

cov_diag_df = pd.DataFrame(cov_diag_rows)
cov_diag_df.to_csv(
    SANITIZATION_OUTPUT_DIR / "covariance_condition_diagnostics_after_sanitization.csv",
    index=False,
)


# ---------------------------------------------------------------------
# 7. Monkey-patch LedoitWolf so downstream cells automatically add ridge
# ---------------------------------------------------------------------

class RidgeRegularizedLedoitWolf:
    """
    Drop-in wrapper for sklearn.covariance.LedoitWolf.

    Downstream notebook cells call LedoitWolf().fit(...), then read
    .covariance_ and .shrinkage_. This wrapper preserves that interface but
    adds an explicit ridge term to covariance_ after Ledoit-Wolf fitting.
    """

    def __init__(self, *args, covariance_ridge: float = CORAL_COVARIANCE_RIDGE, **kwargs):
        self.covariance_ridge = covariance_ridge
        self._estimator = _SklearnLedoitWolf(*args, **kwargs)

    def fit(self, X, y=None):
        self._estimator.fit(X, y)

        cov = np.asarray(self._estimator.covariance_, dtype=np.float64)
        cov = (cov + cov.T) / 2.0

        self.covariance_before_ridge_ = cov.copy()
        self.covariance_ = cov + self.covariance_ridge * np.eye(cov.shape[0])
        self.covariance_ = (self.covariance_ + self.covariance_.T) / 2.0

        # Preserve commonly used attributes.
        self.shrinkage_ = float(self._estimator.shrinkage_)

        # Precision is not used in your current downstream cells, but keep it
        # available for compatibility.
        self.precision_ = np.linalg.pinv(self.covariance_)

        return self

    def __getattr__(self, name):
        return getattr(self._estimator, name)


# Rebind LedoitWolf in the notebook namespace so later cells use ridge regularization.
LedoitWolf = RidgeRegularizedLedoitWolf


# ---------------------------------------------------------------------
# 8. Replace training feature matrices with sanitized covariance-estimation copies
# ---------------------------------------------------------------------

# Preserve originals in case you want to inspect them later in the notebook.
X_source_train_original_for_reference = X_source_train
X_target_train_original_for_reference = X_target_train
feature_order_original_for_reference = feature_order_original

# Downstream cells use X_source_train, X_target_train, and load_feature_order(...).
# Replace them with sanitized matrices so the existing extraction cells remain compatible.
X_source_train = X_source_sanitized
X_target_train = X_target_sanitized

SANITIZED_FEATURE_ORDER = list(features_kept)
shared_features = list(SANITIZED_FEATURE_ORDER)

_original_load_feature_order = load_feature_order

def load_feature_order(path: Path):
    """
    Return sanitized feature order for CORAL stats estimation.

    This intentionally overrides the feature order loaded from disk so later
    CORAL extraction cells use the same feature subset retained after
    winsorization and near-zero variance filtering.
    """
    return list(SANITIZED_FEATURE_ORDER)


# ---------------------------------------------------------------------
# 9. Save sanitization metadata
# ---------------------------------------------------------------------

sanitization_metadata = {
    "winsor_lower_q": WINSOR_LOWER_Q,
    "winsor_upper_q": WINSOR_UPPER_Q,
    "near_zero_variance_threshold": NEAR_ZERO_VARIANCE_THRESHOLD,
    "covariance_ridge": CORAL_COVARIANCE_RIDGE,
    "extreme_max_abs_threshold": EXTREME_MAX_ABS_THRESHOLD,
    "extreme_feature_std_threshold": EXTREME_FEATURE_STD_THRESHOLD,
    "max_condition_number": MAX_CONDITION_NUMBER,
    "original_feature_count": len(feature_order_original),
    "sanitized_feature_count": len(SANITIZED_FEATURE_ORDER),
    "dropped_features": features_dropped,
    "sanitized_feature_order": SANITIZED_FEATURE_ORDER,
    "note": (
        "X_source_train and X_target_train were replaced in-memory with "
        "winsorized, near-zero-variance-filtered copies for CORAL stats estimation. "
        "LedoitWolf was wrapped in-memory to add explicit covariance ridge."
    ),
}

joblib.dump(
    sanitization_metadata,
    SANITIZATION_OUTPUT_DIR / "coral_sanitization_metadata.joblib",
)

with open(SANITIZATION_OUTPUT_DIR / "sanitized_feature_order.json", "w", encoding="utf-8") as f:
    json.dump({"feature_order": SANITIZED_FEATURE_ORDER}, f, indent=2)

print("\nPre-estimation data sanitization complete")
print("=" * 80)
print(f"Original feature count: {len(feature_order_original)}")
print(f"Sanitized feature count: {len(SANITIZED_FEATURE_ORDER)}")
print(f"Dropped feature count: {len(features_dropped)}")
print(f"CORAL covariance ridge: {CORAL_COVARIANCE_RIDGE:.6e}")
print(f"Diagnostics saved to: {SANITIZATION_OUTPUT_DIR}")
print("\nDownstream cells will now use:")
print("  - winsorized source/target copies")
print("  - sanitized feature order")
print("  - ridge-regularized LedoitWolf covariance estimates")

Pre-estimation feature-space check
--------------------------------------------------------------------------------
Original shared feature count: 77
Source features: (1259420, 77)
Target features: (2733516, 77)
Column order identical: True

Source before sanitization diagnostics
--------------------------------------------------------------------------------
shape: (1259420, 77)
global min / max: -1065.671786 / 401.250542
global mean / std: 0.000000 / 0.946628
mean abs / max abs: 0.369405 / 1065.671786
feature std min / median / max: 0.000000 / 1.000000 / 1.000000
near-zero variance features: 8
features over max_abs threshold (1000): 1
features over std threshold (100): 0

Top 10 features by max_abs:
                    feature     max_abs          min        max  std
          Fwd Header Length 1065.671786 -1065.671786   0.155154  1.0
          Bwd Header Length  557.695619  -557.695619   3.034319  1.0
          Subflow Fwd Bytes  401.250542    -0.088172 401.250542  1.0
Total Length 

/var/folders/cz/y2skhn6512n92w71g06v44f00000gn/T/ipykernel_14390/1818990127.py:120: UserWarning: Extreme scaled values were detected before covariance estimation. Flagged rows saved to models/coral_sanitization_diagnostics/flagged_extreme_features_before_sanitization.csv
  warnings.warn(message)



Near-zero variance feature check
--------------------------------------------------------------------------------
Near-zero variance features in source: 14
Near-zero variance features in target: 14
Near-zero variance features in either dataset: 16

Flagged near-zero variance features, not dropped:
             feature  source_variance_after_winsorization  target_variance_after_winsorization
   Bwd Avg Bulk Rate                         0.000000e+00                         0.000000e+00
  Bwd Avg Bytes/Bulk                         0.000000e+00                         0.000000e+00
Bwd Avg Packets/Bulk                         0.000000e+00                         0.000000e+00
   Bwd Header Length                         1.250852e-08                         5.323258e-09
       Bwd PSH Flags                         0.000000e+00                         0.000000e+00
       Bwd URG Flags                         0.000000e+00                         0.000000e+00
      CWE Flag Count               

/var/folders/cz/y2skhn6512n92w71g06v44f00000gn/T/ipykernel_14390/1818990127.py:301: UserWarning: Near-zero variance features were detected. They were NOT dropped, so the covariance matrices keep the expected feature-space shape. Stability will instead be handled using winsorization plus explicit ridge regularization.
  warnings.warn(



Feature retention summary
--------------------------------------------------------------------------------
Original feature count: 77
Features retained for CORAL stats: 77
Features dropped: 0
Covariance matrix shape will remain: 77 x 77

Source after sanitization diagnostics
--------------------------------------------------------------------------------
shape: (1259420, 77)
global min / max: -0.938626 / 7.177239
global mean / std: -0.012829 / 0.708610
mean abs / max abs: 0.356403 / 7.177239
feature std min / median / max: 0.000000 / 0.706194 / 1.000000
near-zero variance features: 14
features over max_abs threshold (1000): 0
features over std threshold (100): 0

Top 10 features by max_abs:
                feature  max_abs       min      max      std
          Fwd Packets/s 7.177239 -0.264308 7.177239 0.997327
         Flow Packets/s 7.027454 -0.277391 7.027454 0.994710
               Idle Std 7.027124 -0.129010 7.027124 0.813976
            Fwd IAT Min 5.966306 -0.119209 5.966306 0.6

In [5]:
### Extract (*GLOBAL) CORAL statistics from source ###
# Exports coral stats artifact to: "models/global_coral_source_stats.joblib"

# Load shared feature space contract.
shared_feature_space_path = Path("data/processed/shared_feature_space.json")
if not shared_feature_space_path.exists():
    raise FileNotFoundError(f"Shared feature space file not found at {shared_feature_space_path}")

# Reuse unified parser so feature-space JSON is handled consistently across cells.
feature_order = load_feature_order(shared_feature_space_path)

# Improvement support: persist canonical feature ordering inside CORAL stats so
# evaluation can verify schema compatibility before adaptation.
X_train_aligned = X_source_train[feature_order]

# Convert to numpy for CORAL math using float64 for better numerical stability.
X_src = X_train_aligned.to_numpy(dtype=np.float64)

# 1) Feature-wise mean vector.
source_feature_mean = np.mean(X_src, axis=0)

# 2) Centered source data.
X_src_centered = X_src - source_feature_mean

# Improvement #1: use Ledoit-Wolf shrinkage covariance for a better-conditioned
# covariance estimate than plain sample covariance.
source_cov_estimator = LedoitWolf()
source_cov_estimator.fit(X_src_centered)
source_covariance = np.asarray(source_cov_estimator.covariance_, dtype=np.float64)
source_covariance = (source_covariance + source_covariance.T) / 2.0

# Save diagnostics consumed by training/eval for spectral-floor and stability analysis.
source_eigenvalues = np.linalg.eigvalsh(source_covariance)
source_min_eig = float(source_eigenvalues.min())
source_max_eig = float(source_eigenvalues.max())
source_cov_condition_number = float(np.linalg.cond(source_covariance))
source_covariance_ridge = float(getattr(source_cov_estimator, "covariance_ridge", 0.0))

# Sanity checks.
assert source_covariance.shape[0] == source_covariance.shape[1], "Covariance matrix must be square"
assert source_covariance.shape[0] == len(feature_order), "Covariance dimension mismatch with feature space"

# Package CORAL statistics.
coral_source_stats = {
    "feature_order": feature_order,
    "mean": source_feature_mean,
    "covariance": source_covariance,
    "covariance_estimator": "LedoitWolf",
    "covariance_shrinkage": float(source_cov_estimator.shrinkage_),
    "min_eigenvalue_before_regularization": source_min_eig,
    "max_eigenvalue": source_max_eig,
    "covariance_condition_number": source_cov_condition_number,
    "covariance_ridge": source_covariance_ridge,
}

# Persist for downstream domain adaptation pipeline.
coral_stats_path = Path("models/global_coral_source_stats.joblib")
coral_stats_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(coral_source_stats, coral_stats_path)

print("CORAL source statistics extracted and saved successfully.")
print(f"Saved to: {coral_stats_path}")
print(f"Features: {len(feature_order)}")
print(f"Covariance shape: {source_covariance.shape}")
print(f"Covariance estimator: LedoitWolf (shrinkage={source_cov_estimator.shrinkage_:.6f})")
print(f"Covariance eigenvalues: min={source_min_eig:.6e}, max={source_max_eig:.6e}")
print(f"Covariance condition number: {source_cov_condition_number:.6e}")

CORAL source statistics extracted and saved successfully.
Saved to: models/global_coral_source_stats.joblib
Features: 77
Covariance shape: (77, 77)
Covariance estimator: LedoitWolf (shrinkage=0.000011)
Covariance eigenvalues: min=1.000543e-02, max=1.474511e+01
Covariance condition number: 1.473711e+03


In [6]:
### Extract (*PER-CLASS) CORAL statistics from source ###
# Exports coral stats artifact to: models/perclass_coral_source_stats.joblib"

perclass_stats = {}
classes_in_train = sorted(pd.Series(y_source_train).unique().tolist())

# Resolve feature order locally so this cell is robust to out-of-order execution.
shared_feature_space_path = Path("data/processed/shared_feature_space.json")
if not shared_feature_space_path.exists():
    raise FileNotFoundError(f"Shared feature space file not found at {shared_feature_space_path}")
feature_order = load_feature_order(shared_feature_space_path)

if len(classes_in_train) == 0:
    raise ValueError("No classes found in y_source_train; cannot compute per-class CORAL statistics")

for class_id in classes_in_train:
    class_mask = (y_source_train == class_id)
    class_count = int(class_mask.sum())
    if class_count < 2:
        raise ValueError(
            f"Class {class_id} has only {class_count} sample(s) in training split. "
            "At least 2 samples are required to estimate covariance."
        )

    # Keep exact same feature ordering contract as global CORAL statistics.
    X_class = X_source_train.loc[class_mask, feature_order]
    X_class_np = X_class.to_numpy(dtype=np.float64)

    # 1) Feature-wise mean vector for this class.
    class_feature_mean = np.mean(X_class_np, axis=0)

    # 2) Center class data.
    X_class_centered = X_class_np - class_feature_mean

    # 3) Ledoit-Wolf covariance estimate (same estimator as global cell).
    class_cov_estimator = LedoitWolf()
    class_cov_estimator.fit(X_class_centered)
    class_covariance = np.asarray(class_cov_estimator.covariance_, dtype=np.float64)
    class_covariance = (class_covariance + class_covariance.T) / 2.0

    # Diagnostics mirroring global-stat extraction.
    class_eigenvalues = np.linalg.eigvalsh(class_covariance)
    class_min_eig = float(class_eigenvalues.min())
    class_max_eig = float(class_eigenvalues.max())
    class_cov_condition_number = float(np.linalg.cond(class_covariance))
    class_covariance_ridge = float(getattr(class_cov_estimator, "covariance_ridge", 0.0))

    assert class_covariance.shape[0] == class_covariance.shape[1], "Covariance matrix must be square"
    assert class_covariance.shape[0] == len(feature_order), "Covariance dimension mismatch with feature space"

    class_name = str(class_id)
    if "le" in globals() and hasattr(le, "classes_"):
        # Prefer original class label when encoder is available.
        class_name = str(le.inverse_transform([int(class_id)])[0])

    perclass_stats[class_name] = {
        "class_id": int(class_id),
        "class_name": class_name,
        "sample_count": class_count,
        "feature_order": feature_order,
        "mean": class_feature_mean,
        "covariance": class_covariance,
        "covariance_estimator": "LedoitWolf",
        "covariance_shrinkage": float(class_cov_estimator.shrinkage_),
        "min_eigenvalue_before_regularization": class_min_eig,
        "max_eigenvalue": class_max_eig,
        "covariance_condition_number": class_cov_condition_number,
        "covariance_ridge": class_covariance_ridge,
    }

# Persist per-class stats for downstream class-conditional CORAL variants.
perclass_stats_path = Path("models/perclass_coral_source_stats.joblib")
perclass_stats_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(perclass_stats, perclass_stats_path)

print("Per-class CORAL source statistics extracted and saved successfully.")
print(f"Saved to: {perclass_stats_path}")
print(f"Classes processed: {len(perclass_stats)}")
for class_name, stats in perclass_stats.items():
    print(
        f"- {class_name}: n={stats['sample_count']}, "
        f"cov_shape={stats['covariance'].shape}, "
        f"shrinkage={stats['covariance_shrinkage']:.6f}, "
        f"min_eig={stats['min_eigenvalue_before_regularization']:.6e}, "
        f"cond={stats['covariance_condition_number']:.6e}"
    )

Per-class CORAL source statistics extracted and saved successfully.
Saved to: models/perclass_coral_source_stats.joblib
Classes processed: 8
- Benign: n=944565, cov_shape=(77, 77), shrinkage=0.000034, min_eig=1.001059e-02, cond=5.567672e+02
- DDoS: n=102420, cov_shape=(77, 77), shrinkage=0.000022, min_eig=1.001156e-02, cond=2.373872e+03
- DoS GoldenEye: n=8234, cov_shape=(77, 77), shrinkage=0.000306, min_eig=1.014576e-02, cond=2.715376e+03
- DoS Hulk: n=184099, cov_shape=(77, 77), shrinkage=0.000012, min_eig=1.000847e-02, cond=3.422039e+03
- DoS Slowhttptest: n=4399, cov_shape=(77, 77), shrinkage=0.000893, min_eig=1.072887e-02, cond=2.712025e+03
- DoS slowloris: n=4637, cov_shape=(77, 77), shrinkage=0.000441, min_eig=1.045493e-02, cond=3.962646e+03
- FTP-Patator: n=6348, cov_shape=(77, 77), shrinkage=0.000115, min_eig=1.002708e-02, cond=1.602328e+03
- SSH-Patator: n=4718, cov_shape=(77, 77), shrinkage=0.000052, min_eig=1.000511e-02, cond=7.319054e+02


In [7]:
### Extract (*GLOBAL) CORAL statistics from target ###
# Exports coral stats artifact to: "models/global_coral_target_stats.joblib"

# Load shared feature space contract.
shared_feature_space_path = Path("data/processed/shared_feature_space.json")
if not shared_feature_space_path.exists():
    raise FileNotFoundError(f"Shared feature space file not found at {shared_feature_space_path}")

# Reuse unified parser so feature-space JSON is handled consistently across cells.
feature_order = load_feature_order(shared_feature_space_path)

# Improvement support: persist canonical feature ordering inside CORAL stats so
# evaluation can verify source/target schema parity before adaptation.
X_target_train_aligned = X_target_train[feature_order]

# Convert to numpy for CORAL math using float64 for better numerical stability.
X_trg = X_target_train_aligned.to_numpy(dtype=np.float64)

# 1) Feature-wise mean vector.
target_feature_mean = np.mean(X_trg, axis=0)

# 2) Centered target training data.
X_trg_centered = X_trg - target_feature_mean

# Improvement #1: use Ledoit-Wolf shrinkage covariance for a better-conditioned
# covariance estimate than plain sample covariance.
target_cov_estimator = LedoitWolf()
target_cov_estimator.fit(X_trg_centered)
target_covariance = np.asarray(target_cov_estimator.covariance_, dtype=np.float64)
target_covariance = (target_covariance + target_covariance.T) / 2.0

# Save diagnostics consumed by training/eval for spectral-floor and stability analysis.
target_eigenvalues = np.linalg.eigvalsh(target_covariance)
target_min_eig = float(target_eigenvalues.min())
target_max_eig = float(target_eigenvalues.max())
target_cov_condition_number = float(np.linalg.cond(target_covariance))
target_covariance_ridge = float(getattr(target_cov_estimator, "covariance_ridge", 0.0))

# Sanity checks.
assert target_covariance.shape[0] == target_covariance.shape[1], "Covariance matrix must be square"
assert target_covariance.shape[0] == len(feature_order), "Covariance dimension mismatch with feature space"

# Package CORAL statistics.
coral_target_stats = {
    "feature_order": feature_order,
    "mean": target_feature_mean,
    "covariance": target_covariance,
    "covariance_estimator": "LedoitWolf",
    "covariance_shrinkage": float(target_cov_estimator.shrinkage_),
    "min_eigenvalue_before_regularization": target_min_eig,
    "max_eigenvalue": target_max_eig,
    "covariance_condition_number": target_cov_condition_number,
    "covariance_ridge": target_covariance_ridge,
}

# Persist for downstream domain adaptation pipeline.
coral_stats_path = Path("models/global_coral_target_stats.joblib")
coral_stats_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(coral_target_stats, coral_stats_path)

print("CORAL target statistics extracted and saved successfully.")
print("Verified: CORAL stats were computed from target train split only.")
print(f"Train rows used for CORAL: {len(X_target_train_aligned)}")
print(f"Saved to: {coral_stats_path}")
print(f"Features: {len(feature_order)}")
print(f"Covariance shape: {target_covariance.shape}")
print(f"Covariance estimator: LedoitWolf (shrinkage={target_cov_estimator.shrinkage_:.6f})")
print(f"Covariance eigenvalues: min={target_min_eig:.6e}, max={target_max_eig:.6e}")
print(f"Covariance condition number: {target_cov_condition_number:.6e}")

CORAL target statistics extracted and saved successfully.
Verified: CORAL stats were computed from target train split only.
Train rows used for CORAL: 2733516
Saved to: models/global_coral_target_stats.joblib
Features: 77
Covariance shape: (77, 77)
Covariance estimator: LedoitWolf (shrinkage=0.000001)
Covariance eigenvalues: min=1.002142e-02, max=1.156748e+03
Covariance condition number: 1.154276e+05


In [8]:
### Extract (*PER-CLASS) CORAL statistics from target ###
# Exports coral stats artifact to: "models/perclass_coral_target_stats.joblib"

perclass_stats = {}
classes_in_train = sorted(pd.Series(y_target_train).unique().tolist())

# Resolve feature order locally so this cell is robust to out-of-order execution.
shared_feature_space_path = Path("data/processed/shared_feature_space.json")
if not shared_feature_space_path.exists():
    raise FileNotFoundError(f"Shared feature space file not found at {shared_feature_space_path}")
feature_order = load_feature_order(shared_feature_space_path)

if len(classes_in_train) == 0:
    raise ValueError("No classes found in y_target_train; cannot compute per-class CORAL statistics")

for class_id in classes_in_train:
    class_mask = (y_target_train == class_id)
    class_count = int(class_mask.sum())
    if class_count < 2:
        raise ValueError(
            f"Class {class_id} has only {class_count} sample(s) in training split. "
            "At least 2 samples are required to estimate covariance."
        )

    # Keep exact same feature ordering contract as global CORAL statistics.
    X_class = X_target_train.loc[class_mask, feature_order]
    X_class_np = X_class.to_numpy(dtype=np.float64)

    # 1) Feature-wise mean vector for this class.
    class_feature_mean = np.mean(X_class_np, axis=0)

    # 2) Center class data.
    X_class_centered = X_class_np - class_feature_mean

    # 3) Ledoit-Wolf covariance estimate (same estimator as global cell).
    class_cov_estimator = LedoitWolf()
    class_cov_estimator.fit(X_class_centered)
    class_covariance = np.asarray(class_cov_estimator.covariance_, dtype=np.float64)
    class_covariance = (class_covariance + class_covariance.T) / 2.0

    # Diagnostics mirroring global-stat extraction.
    class_eigenvalues = np.linalg.eigvalsh(class_covariance)
    class_min_eig = float(class_eigenvalues.min())
    class_max_eig = float(class_eigenvalues.max())
    class_cov_condition_number = float(np.linalg.cond(class_covariance))
    class_covariance_ridge = float(getattr(class_cov_estimator, "covariance_ridge", 0.0))

    assert class_covariance.shape[0] == class_covariance.shape[1], "Covariance matrix must be square"
    assert class_covariance.shape[0] == len(feature_order), "Covariance dimension mismatch with feature space"

    class_name = str(class_id)
    if "le" in globals() and hasattr(le, "classes_"):
        # Prefer original class label when encoder is available.
        class_name = str(le.inverse_transform([int(class_id)])[0])

    perclass_stats[class_name] = {
        "class_id": int(class_id),
        "class_name": class_name,
        "sample_count": class_count,
        "feature_order": feature_order,
        "mean": class_feature_mean,
        "covariance": class_covariance,
        "covariance_estimator": "LedoitWolf",
        "covariance_shrinkage": float(class_cov_estimator.shrinkage_),
        "min_eigenvalue_before_regularization": class_min_eig,
        "max_eigenvalue": class_max_eig,
        "covariance_condition_number": class_cov_condition_number,
        "covariance_ridge": class_covariance_ridge,
    }

# Persist per-class stats for downstream class-conditional CORAL variants.
perclass_stats_path = Path("models/perclass_coral_target_stats.joblib")
perclass_stats_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(perclass_stats, perclass_stats_path)

print("Per-class CORAL target statistics extracted and saved successfully.")
print(f"Saved to: {perclass_stats_path}")
print(f"Classes processed: {len(perclass_stats)}")
for class_name, stats in perclass_stats.items():
    print(
        f"- {class_name}: n={stats['sample_count']}, "
        f"cov_shape={stats['covariance'].shape}, "
        f"shrinkage={stats['covariance_shrinkage']:.6f}, "
        f"min_eig={stats['min_eigenvalue_before_regularization']:.6e}, "
        f"cond={stats['covariance_condition_number']:.6e}"
    )

Per-class CORAL target statistics extracted and saved successfully.
Saved to: models/perclass_coral_target_stats.joblib
Classes processed: 8
- Benign: n=2050137, cov_shape=(77, 77), shrinkage=0.000001, min_eig=1.002463e-02, cond=1.288503e+05
- DDoS: n=161384, cov_shape=(77, 77), shrinkage=0.000002, min_eig=1.005231e-02, cond=2.029389e+05
- DoS GoldenEye: n=33206, cov_shape=(77, 77), shrinkage=0.000364, min_eig=1.014480e-02, cond=2.203990e+03
- DoS Hulk: n=240000, cov_shape=(77, 77), shrinkage=0.000250, min_eig=1.000348e-02, cond=6.052030e+01
- DoS Slowhttptest: n=80000, cov_shape=(77, 77), shrinkage=0.000038, min_eig=1.001165e-02, cond=2.342285e+03
- DoS slowloris: n=8792, cov_shape=(77, 77), shrinkage=0.000392, min_eig=1.048463e-02, cond=5.178390e+03
- FTP-Patator: n=79997, cov_shape=(77, 77), shrinkage=0.000012, min_eig=1.001421e-02, cond=9.307044e+03
- SSH-Patator: n=80000, cov_shape=(77, 77), shrinkage=0.000005, min_eig=1.000086e-02, cond=1.315182e+03
